In [1]:
import pandas as pd
import numpy as np
import commodity_common_functions as CCF
import seaborn as sns
import dataframe_image as dfi
from pathlib import Path
import json
import seaborn as sns
import datetime as dt
import logging
from datetime import datetime
import os
import OptionRiskMetrixETF as ORME
import OptionRiskMetrix as ORM

In [2]:
def findAllInfo(date = None):
    try:
        if(date == None):
            date = dt.date.today().strftime("%Y%m%d")
            date = CCF.tradingDay[CCF.tradingDay < date][-1]
        
        cinfo = {}
        etfinfo = {}
        
        try:
            cinfo = findUnderlyingOptionInfoCommodity(date)
        except Exception as e:
            logging.error(f"获取商品期权信息失败: {e}")
        
        try:
            etfinfo = findUnderlyingOptionInfoETF(date)
        except Exception as e:
            logging.error(f"获取ETF期权信息失败: {e}")
            
        return {**cinfo, **etfinfo}
    except Exception as e:
        logging.error(f"findAllInfo函数执行失败: {e}")
        return {}
    

def findUnderlyingOptionInfoCommodity(date = None):
    if(date == None):
        date = dt.date.today().strftime("%Y%m%d")
        date = CCF.tradingDay[CCF.tradingDay < date][-1]
    SQL_cmd = f"""
    SELECT underlying_instr_id,instrument_id
    FROM daily_data_option
    WHERE trading_date = '{date}'
    ORDER BY open_interest DESC
    """
    max_open_interest = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    max_open_interest = (
        max_open_interest.groupby("underlying_instr_id").first().reset_index()
    )
    max_open_interest["underlying_instr_id"] = max_open_interest[
        "underlying_instr_id"
    ].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )
    SQL_cmd = f"""
    SELECT         
        i.underlying_instr_id,
        i.exchange_id,
        i.expiredate,
        i.volume_multiple as option_multiple,
        i.price_tick,
        icr.open_money_by_vol
    FROM instrument i
    LEFT JOIN commission_info icr ON i.instrument_id = icr.instrument_id
    WHERE i.instrument_id IN {tuple(max_open_interest['instrument_id'])}
    """
    instrument_info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    instrument_info["underlying_instr_id"] = instrument_info[
        "underlying_instr_id"
    ].apply(
        lambda x: (
            "IH" + x[2:]
            if x.startswith("HO")
            else (
                "IF" + x[2:]
                if x.startswith("IO")
                else ("IM" + x[2:] if x.startswith("MO") else x)
            )
        )
    )
    instrument_info["open_money_by_vol"] = np.where(
        instrument_info["underlying_instr_id"].str.startswith(("IH", "IF", "IM")),
        15,
        instrument_info["open_money_by_vol"],
    )
    SQL_cmd = f"""
    SELECT instrument_id as underlying_instr_id, short_margin_ratio as margin_ratio, product_id as product
    FROM instrument
    WHERE instrument_id IN {tuple(max_open_interest['underlying_instr_id'])}
    """
    margin_ratio = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    SQL_cmd = f"""
    SELECT PRODUCT_ID as product, WIND_INDUSTRYNAME1 as sector
    FROM WindIndustry
    WHERE PRODUCT_ID IN {tuple(margin_ratio['product'])}
    """
    secinfo = pd.read_sql(sql=SQL_cmd, con=CCF.research)
    secinfo.loc[len(secinfo)] = ["IH", "金融"]
    secinfo.loc[len(secinfo)] = ["IF", "金融"]
    secinfo.loc[len(secinfo)] = ["IC", "金融"]
    secinfo.loc[len(secinfo)] = ["IM", "金融"]
    SQL_cmd = f"""
    SELECT         
    instrument_id,
    prev_settlement,
    limit_up
    FROM daily_data
    WHERE trading_date = '{date[:4]}-{date[4:6]}-{date[6:]}'
    """
    limitupdf = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    limitupdf["updownlimit"] = (
        limitupdf["limit_up"] / limitupdf["prev_settlement"] - 1
    ).round(2)
    limitupdf = limitupdf[["instrument_id", "updownlimit"]]
    limitupdf = limitupdf.rename(columns={"instrument_id": "underlying_instr_id"})

    merged_df = pd.merge(
        max_open_interest, instrument_info, on="underlying_instr_id", how="left"
    )
    merged_df = pd.merge(merged_df, margin_ratio, on="underlying_instr_id", how="left")
    merged_df = pd.merge(merged_df, limitupdf, on="underlying_instr_id", how="left")
    merged_df = pd.merge(merged_df, secinfo, on="product", how="left")
    merged_df = merged_df.drop(columns=["instrument_id"]).dropna()
    merged_dict = merged_df.set_index(["underlying_instr_id", "expiredate"]).T.to_dict("dict")
    for key in merged_dict:
        merged_dict[key]["underlying_instr_id"] = key[0]
        merged_dict[key]["expiredate"] = key[1]
        if merged_dict[key]["exchange_id"] == "CFFEX":
            merged_dict[key]["exchange_id"] = "CFE"
        elif merged_dict[key]["exchange_id"] == "SHFE":
            merged_dict[key]["exchange_id"] = "SHF"
        elif merged_dict[key]["exchange_id"] == "CZCE":
            merged_dict[key]["exchange_id"] = "CZC"
        elif merged_dict[key]["exchange_id"] == "GFEX":
            merged_dict[key]["exchange_id"] = "GFE"
        if key[:2] == 'IH':
            merged_dict[key]["underlying_multiple"] = 300
        elif key[:2] == 'IF':
            merged_dict[key]["underlying_multiple"] = 300
        elif key[:2] == 'IM':
            merged_dict[key]["underlying_multiple"] = 200
        else:
            merged_dict[key]["underlying_multiple"] = merged_dict[key]["option_multiple"]
    return merged_dict

def findUnderlyingOptionInfoETF(date = None):
    if(date == None):
        date = dt.date.today().strftime("%Y%m%d")
        date = CCF.tradingDay[CCF.tradingDay < date][-1]
    SQL_cmd = f"""SELECT instrument_id
    FROM daily_data_ut
    WHERE trading_date = '{date}'
    """
    df = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
    df = df[df["instrument_id"].str.len() > 6]
    SQL_cmd = f"""
    SELECT underlying_instr_id, exchange_id, expiredate, volume_multiple as option_multiple, price_tick
    FROM instrument
    WHERE instrument_id IN {tuple(df['instrument_id'])}
    """
    info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    info = info.drop_duplicates(subset=["underlying_instr_id", "expiredate"])
    info["sector"] = "金融"
    info["open_money_by_vol"] = 0
    info["updownlimit"] = 0.1
    info["margin_ratio"] = 0.12
    info["underlying_multiple"] = 10000
    merged_dict = info.set_index(["underlying_instr_id", "expiredate"]).T.to_dict("dict")
    for key in merged_dict:
        merged_dict[key]["underlying_instr_id"] = key[0]
        merged_dict[key]["expiredate"] = key[1]
        if merged_dict[key]["exchange_id"] == "SZSE":
            merged_dict[key]["exchange_id"] = "SZ"
        else:
            merged_dict[key]["exchange_id"] = "SH"

        if key[0] == "510300":
            merged_dict[key]["product"]  = "(华)沪深300"
        elif key[0] == "510500":
            merged_dict[key]["product"]  = "(南)中证500"
        elif key[0] == "510050":
            merged_dict[key]["product"]  = "(华)上证50"
        elif key[0] == "159915":
            merged_dict[key]["product"]  = "(易)创业板"
        elif key[0] == "588000":
            merged_dict[key]["product"]  = "(华)科创50"
        elif key[0] == "159901":
            merged_dict[key]["product"]  = "(易)深证100"
    return merged_dict

In [3]:
def findInstrumentPrice(instrument, date):
    """
    根据给定的合约ID和日期查询收盘价
    
    参数:
        instrument: 合约ID
        date: 日期字符串，格式为'YYYYMMDD'
    
    返回:
        收盘价，如果未找到则返回-1
    """
    try:
        # 转换日期格式为'YYYY-MM-DD'用于daily_data表
        bardate = f"{date[:4]}-{date[4:6]}-{date[6:8]}"
        
        # 依次在三个表中查询
        tables = [
            ("daily_data", "trading_date", bardate),
            ("daily_data_option", "trading_date", date),
            ("daily_data_ut", "trading_date", date)
        ]
        
        for table, date_col, date_val in tables:
            SQL_cmd = f"""
            SELECT close
            FROM {table}
            WHERE instrument_id = '{instrument}' AND {date_col} = '{date_val}'
            """
            info = pd.read_sql(sql=SQL_cmd, con=CCF.std_market_data)
            if len(info) > 0:
                return info.iloc[0]['close']
        
        # 如果所有表中都没有找到数据
        logging.error(f"未找到合约 {instrument} 在日期 {date} 的收盘价")
        return np.nan
    except Exception as e:
        logging.error(f"findInstrumentPrice函数执行失败: {str(e)}")
        return np.nan
    
option_quotesave = {}
def ETF_Option_Raw_Data(wind_code: str, tradingdate: str):
    if (wind_code, tradingdate) in option_quotesave:
        return option_quotesave[(wind_code, tradingdate)]
    """

    :param wind_code:
    :param tradingdate:
    :return: dataframe 包含字段 id, seq_no, Bid1, Ask1, MidPrice
    """
    future_path = CCF.Ftp_Path(tradingdate)
    option_data = f"{future_path}ut_std_data/{tradingdate}/{wind_code}.csv"
    # print(option_data)
    OptionData = pd.read_csv(
        option_data,
        usecols=["id", "seq_no", "bid_price1", "ask_price1"],
        dtype={
            "id": object,
            "seq_no": np.int64,
            "bid_price1": np.float64,
            "ask_price1": np.float64,
        },
        escapechar="/",
        na_values=r"\N",
    )

    OptionData.rename(
        columns={"bid_price1": "Bid1", "ask_price1": "Ask1"}, inplace=True
    )
    OptionData.loc[:, "MidPrice"] = 0.5 * (OptionData["Bid1"] + OptionData["Ask1"])
    option_quotesave[(wind_code, tradingdate)] = OptionData
    return OptionData

def Option_Raw_Data(wind_code: str, tradingdate: str):
    if (wind_code, tradingdate) in option_quotesave:
        return option_quotesave[(wind_code, tradingdate)]
    """

    :param wind_code:
    :param tradingdate:
    :return: dataframe 包含字段 id, seq_no, Bid1, Ask1, MidPrice
    """
    future_path = CCF.Ftp_Path(tradingdate)
    option_data = f"{future_path}ctp_std_data/{tradingdate}/{wind_code}.csv"
    # print(option_data)
    OptionData = pd.read_csv(
        option_data,
        usecols=["id", "seq_no", "bid_price1", "ask_price1"],
        dtype={
            "id": object,
            "seq_no": np.int64,
            "bid_price1": np.float64,
            "ask_price1": np.float64,
        },
        escapechar="/",
        na_values=r"\N",
    )

    OptionData.rename(
        columns={"bid_price1": "Bid1", "ask_price1": "Ask1"}, inplace=True
    )
    OptionData.loc[:, "MidPrice"] = 0.5 * (OptionData["Bid1"] + OptionData["Ask1"])
    option_quotesave[(wind_code, tradingdate)] = OptionData
    return OptionData
    
def findOptionQuote(instrument, date):
    """
    获取期权报价数据
    
    Args:
        instrument: 期权合约代码
        date: 交易日期
        
    Returns:
        tuple: (卖一价, 买一价)，如果获取失败则返回(None, None)
    """
    import time
    start_time = time.time()
    try:
        # 首先尝试使用标准期权数据接口
        row = Option_Raw_Data(instrument, date).iloc[-601]
        result = float(row["Ask1"]), float(row["Bid1"])
    except Exception as e1:
        try:
            # 如果标准接口失败，尝试使用ETF期权数据接口
            row = ETF_Option_Raw_Data(instrument, date).iloc[-601]
            result = float(row["Ask1"]), float(row["Bid1"])
        except Exception as e2:
            logging.error(f"获取期权报价失败 - 合约: {instrument}, 日期: {date}, 错误: {str(e2)}")
            result = np.nan, np.nan
    
    end_time = time.time()
    execution_time = end_time - start_time
    # print(f"findOptionQuote执行时间: {execution_time:.6f}秒 - 合约: {instrument}, 日期: {date}")
    logging.info(f"findOptionQuote执行时间: {execution_time:.6f}秒 - 合约: {instrument}, 日期: {date}")
    
    return result

In [4]:
optionINFO = findAllInfo()


class OptionPortfolio(object):
    def __init__(self, task_name, portfolio, today):
        ### check if all instrument in portfolio have same underlying and expiredate
        
        try:
            futurelst = [i for i in portfolio]
            # 处理只有一个元素的情况，避免SQL语法错误
            if len(futurelst) == 1:
                SQL_cmd = f"""
                SELECT instrument_id, underlying_instr_id, expiredate, options_type, strike_price
                FROM instrument
                WHERE instrument_id = '{futurelst[0]}'
                """
            else:
                SQL_cmd = f"""
                SELECT instrument_id, underlying_instr_id, expiredate, options_type, strike_price
                FROM instrument
                WHERE instrument_id IN {tuple(futurelst)}
                """
            info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
            if len(info) != len(portfolio):
                raise Exception("instrument库中找不到期权信息", futurelst)
            if (
                len(info.drop_duplicates(subset=["underlying_instr_id", "expiredate"]))
                > 1
            ):
                raise Exception(
                    "期权组合中存在多个标的和到期日", futurelst
                )
            info["options_type"] = info["options_type"].apply(
                lambda x: "c" if x == '1' else "p"
            )
            # if(len(info) == 0):
            #     raise Exception('Error, no data found for the given instrument')

            self.underlying_instr_id = info.iloc[0]["underlying_instr_id"]
            if(self.underlying_instr_id[:2] == 'HO'):
                self.underlying_instr_id = 'IH' + self.underlying_instr_id[2:]
            elif(self.underlying_instr_id[:2] == 'IO'):
                self.underlying_instr_id = 'IF' + self.underlying_instr_id[2:]
            elif(self.underlying_instr_id[:2] == 'MO'):
                self.underlying_instr_id = 'IM' + self.underlying_instr_id[2:]
            self.expiredate = info.iloc[0]["expiredate"]
            if (self.underlying_instr_id, self.expiredate) not in optionINFO:
                raise Exception("Error, portfolio has no basic info", self.underlying_instr_id, self.expiredate)

            self.taskname = task_name
            self.Undinfo = optionINFO[(self.underlying_instr_id, self.expiredate)]
            self.portfolio = portfolio
            info = info.set_index("instrument_id")
            self.portInfo = info.to_dict("index")
            self.today = today
            self.sell_side = info.iloc[0]["options_type"]
            self.cdtm = (
                datetime.strptime(self.expiredate, "%Y%m%d")
                - datetime.strptime(today, "%Y%m%d")
            ).days
            self.tdtm = len(
                CCF.tradingDay[
                    (CCF.tradingDay > today) & (CCF.tradingDay <= self.expiredate)
                ]
            )
            self.und_price = findInstrumentPrice(self.underlying_instr_id, today)
            if np.isnan(self.und_price):
                # logging.error(
                #     f"No Daily Close Found in {today}-{self.underlying_instr_id}"
                # )
                raise Exception(f"无标的价格: {self.underlying_instr_id}")
            margin = 0
            for option_id, shares in self.portfolio.items():
                optclose = findInstrumentPrice(option_id, today)
                optask, optbid = findOptionQuote(option_id, today)
                # multiplier = self.Undinfo['option_multiple']
                if np.isnan(optclose) and np.isnan(optask) and np.isnan(optbid):
                    logging.error(f"无法获取期权数据: 日期={date}, 合约={option_id}, 取做小tick")
                    #### 如果期权盘口价格数据缺失，则使用最小tick进行后续计算
                    optask = optbid = optclose = self.Undinfo['price_tick']
                    # raise Exception(f"期权价格数据缺失: {option_id}")
                elif np.isnan(optask) or np.isnan(optbid):
                    logging.error(f"无法获取期权盘口数据: 日期={date}, 合约={option_id}, 取close")
                    optask = optbid = optclose
                    # raise Exception(f"期权价格数据缺失: {option_id}")
                self.portInfo[option_id]["position"] = shares
                self.portInfo[option_id]["ask"] = optask
                self.portInfo[option_id]["bid"] = optbid
                self.portInfo[option_id]["close"] = optclose
                if(shares < 0):
                    otmvalue = (
                        self.und_price * self.Undinfo['option_multiple'] - self.portInfo[option_id]["strike_price"] * self.Undinfo['option_multiple']
                        if self.portInfo[option_id]["options_type"] == "p"
                        else self.portInfo[option_id]["strike_price"] * self.Undinfo['option_multiple'] - self.und_price * self.Undinfo['option_multiple']
                    )
                    otmvalue = max(otmvalue, 0)
                    orimargin = self.und_price * self.Undinfo['option_multiple'] * self.Undinfo['margin_ratio']
                    thismargin = self.portInfo[option_id]["close"] * self.Undinfo['option_multiple'] + max(
                        orimargin - 0.5 * otmvalue, 0.5 * orimargin
                    )
                    margin += thismargin * shares * -1
            self.margin = margin * 1.2
                
        except Exception as e:
            logging.error(f"初始化期权组合失败: {str(e)}")
            raise

    def calMMPortfolioPrice(self):
        portfolio_price = 0
        portfolio_ask = 0
        portfolio_bid = 0
        multiplier = self.Undinfo["option_multiple"]
        for option_id, info in self.portInfo.items():
            shares = info["position"]
            optclose = info["close"]
            optask = info["ask"]
            optbid = info["bid"]
            portfolio_price += shares * optclose * multiplier
            if shares > 0:
                portfolio_ask += shares * optask * multiplier
                portfolio_bid += shares * optbid * multiplier
            else:
                portfolio_bid += shares * optask * multiplier
                portfolio_ask += shares * optbid * multiplier
        # else:
        #     for option_id, shares in self.portfolio.items():
        #         typ = self.portInfo[option_id]['options_type']
        #         k = self.portInfo[option_id]['strike_price']
        #         tdm = len(CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= self.expiredate)])
        #         optclose = CCF.BSMiv.black_scholes_merton(typ, und_price, k, tdm/243, 0, iv, 0)
        #         portfolio_price += shares * optclose * multiplier
        #     portfolio_ask = portfolio_price
        #     portfolio_bid = portfolio_price
        return float(portfolio_price), float(portfolio_ask), float(portfolio_bid)

    def calMTMpnl(self, cost):
        portfolio_price, portfolio_ask, portfolio_bid = self.calMMPortfolioPrice()
        # cost = cost * self.Undinfo["option_multiple"]
        return portfolio_price - cost, portfolio_bid - cost

    def calMTMreturn(self, cost, margin_used):
        pnl, _ = self.calMTMpnl(cost)
        return pnl / margin_used

    def calMTMGreeksInfo(self, method = 'tdtm'):
        """
        计算期权组合的希腊字母信息
        返回包含每个期权合约IV和希腊字母值的字典
        """
        # 检查method参数是否有效
        if method not in ['tdtm', 'cdtm']:
            raise ValueError(f"无效的method参数: {method}，必须是'tdtm'或'cdtm'")
        result_info = {}
        und_close = self.und_price
        if(method == 'tdtm'):
            ttm = self.tdtm/243
        else:
            ttm = self.cdtm/365
        
        for option_id, option_data in self.portInfo.items():
            optclose = option_data["close"]
            optask = option_data["ask"]
            optbid = option_data["bid"]            
            # 确定用于计算希腊字母的合理价格
            if(optask == None or optbid == None):
                price = optclose
            elif optbid < optclose < optask:
                price = optclose
            else:
                price = (optask + optbid)/2
                
            # 计算IV和希腊字母
            iv, delta, gamma, vega, theta = CCF.IV_GREEK(
                price, 
                und_close, 
                option_data['strike_price'], 
                ttm, 
                0, 0, 
                option_data['options_type']
            )
            
            if(method != 'tdtm'):
                theta = theta * 243 / 365
            
            # 存储计算结果
            result_info[option_id] = option_data.copy()
            result_info[option_id].update({
                'iv': iv,
                'delta': delta,
                'gamma': gamma,
                'vega': vega,
                'theta': theta,
                'method': method
            })
            
        return result_info
    
    ######### need  to input info with greeks 
    def calPortfolioDelta(self, info):
        # info = self.calMTMGreeksInfo()
        delta = 0
        for option_id, option_data in info.items():
            delta += option_data['delta'] * option_data['position'] * self.Undinfo['option_multiple']
        return delta, delta*self.und_price, float(delta/self.Undinfo['underlying_multiple'])
    
    def calPortfolioPrice(self, info):
        # info = self.calMTMGreeksInfo()
        price = 0
        for option_id, option_data in info.items():
            optclose = option_data["close"]
            optask = option_data["ask"]
            optbid = option_data["bid"]
            
            # 确定用于计算希腊字母的合理价格
            if optbid < optclose < optask:
                thisp = optclose
            else:
                thisp = (optask + optbid)/2
            price += thisp * option_data['position'] * self.Undinfo['option_multiple']
        return float(price)
    
    def calABSPortInfoInScenrio(self, ttm, undPrice, ivMultiple, method = 'tdtm'):
                #     for option_id, shares in self.portfolio.items():
        #         typ = self.portInfo[option_id]['options_type']
        #         k = self.portInfo[option_id]['strike_price']
        #         tdm = len(CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= self.expiredate)])
        #         optclose = CCF.BSMiv.black_scholes_merton(typ, und_price, k, tdm/243, 0, iv, 0)
        if method not in ['tdtm', 'cdtm']:
            raise Exception("method must be tdtm or cdtm")
        info = self.calMTMGreeksInfo(method)
        result_info = {}
        # first_key = next(iter(info))
        for option_id, option_data in info.items():
            typ = option_data['options_type']
            k = option_data['strike_price']
            iv = option_data['iv'] * ivMultiple
            optclose = CCF.BSMiv.black_scholes_merton(typ, undPrice, k, ttm, 0, iv, 0)
            result_info[option_id] = option_data.copy()
            iv, delta, gamma, vega, theta = CCF.IV_GREEK(
                optclose, 
                undPrice, 
                k, 
                ttm, 
                0, 0, 
                typ
            )
            if(method != 'tdtm'):
                theta = theta * 243 / 365
            result_info[option_id].update({'close': optclose, 'bid': optclose, 'ask': optclose, 'delta': delta, 'gamma': gamma, 'vega': vega, 'theta': theta})
        return result_info
    
    def calRELPortInfoInScenrio(self, dt, smulti, ivMultiple, method = 'tdtm'):
                #     for option_id, shares in self.portfolio.items():
        #         typ = self.portInfo[option_id]['options_type']
        #         k = self.portInfo[option_id]['strike_price']
        #         tdm = len(CCF.tradingDay[(CCF.tradingDay > date) & (CCF.tradingDay <= self.expiredate)])
        #         optclose = CCF.BSMiv.black_scholes_merton(typ, und_price, k, tdm/243, 0, iv, 0)
        if method not in ['tdtm', 'cdtm']:
            raise Exception("method must be tdtm or cdtm")
        info = self.calMTMGreeksInfo(method)
        result_info = {}
        undPrice = self.und_price * smulti
        if(method == 'tdtm'):
            ttm = (self.tdtm - 0.5 - dt + 1)/243
        else:
            ttm = (self.cdtm - 0.5 - dt + 1)/365
            # nextday = CCF.tradingDay[CCF.tradingDay > self.today][dt-1]
            # ttm = ((
            #     datetime.strptime(self.expiredate, "%Y%m%d")
            #     - datetime.strptime(nextday, "%Y%m%d")
            # ).days + 0.5)/365

        # first_key = next(iter(info))
        for option_id, option_data in info.items():
            typ = option_data['options_type']
            k = option_data['strike_price']
            iv = option_data['iv'] * ivMultiple
            optclose = CCF.BSMiv.black_scholes_merton(typ, undPrice, k, ttm, 0, iv, 0)
            result_info[option_id] = option_data.copy()
            iv, delta, gamma, vega, theta = CCF.IV_GREEK(
                optclose, 
                undPrice, 
                k, 
                ttm, 
                0, 0, 
                typ
            )
            if(method != 'tdtm'):
                theta = theta * 243 / 365
            result_info[option_id].update({'close': optclose, 'bid': optclose, 'ask': optclose, 'delta': delta, 'gamma': gamma, 'vega': vega, 'theta': theta})
        return result_info
    
    # #### need to make method the same as info
    # def forwardScenrioAnalysis(self, cost, margin, dt = 1, method = 'tdtm'):
    #     # 从info中获取第一个键，并从其值中获取method
    #     info = self.calMTMGreeksInfo(method)
    #     cost = cost * self.Undinfo["option_multiple"]
    #     first_key = next(iter(info))
    #     # method = info[first_key]['method']
    #     direc = self.sell_side
    #     Slevel = [1,2,3,4]
    #     Schange = np.linspace(0,self.Undinfo['updownlimit'],5)[1:]
    #     if(direc == 'p'):
    #         Slst = (1 - Schange ) * self.und_price
    #     else:
    #         Slst = (1 + Schange ) * self.und_price
    #     ivlevel = [1,2,3,4]
    #     ivchange = [1.1, 1.2, 1.3, 1.4]
    #     # if(method == 'tdtm'):
    #     #     ttm = (self.tdtm - 0.5 - dt + 1)/243
    #     # else:
    #     #     nextday = CCF.tradingDay[CCF.tradingDay > self.today][dt-1]
    #     #     ttm = ((
    #     #         datetime.strptime(self.expiredate, "%Y%m%d")
    #     #         - datetime.strptime(nextday, "%Y%m%d")
    #     #     ).days + 0.5)/365
            
    #     result = []
    #     tempiv =  info[first_key]['iv']
    #     for i in Slevel:
    #         for j in ivlevel:
    #             thiss = Slst[i-1]
    #             thisiv = ivchange[j-1]
    #             thisinfo = self.calRELPortInfoInScenrio(dt, smulti=thiss, ivMultiple=thisiv, method=method)
    #             thisprice = self.calPortfolioPrice(thisinfo)
    #             result.append([i, j, thiss, tempiv*thisiv, thisprice])
    #     result = pd.DataFrame(result, columns=['Slevel', 'ivlevel', 'S', 'iv', 'price'])
    #     result['PNL'] = result['price'] - cost
    #     result['RET'] = result['PNL'] / margin
    #     return result
    
    def backwardScenrioAnalysisLevelS(self, margin, LGDratio, ivlevel = 1, dt = 1, method = 'tdtm'):
        if method not in ['tdtm', 'cdtm']:
            raise Exception("method must be tdtm or cdtm")
        # cost = cost * self.Undinfo["option_multiple"]
        info = self.calMTMGreeksInfo(method)
        # first_key = next(iter(info))
        # method = info[first_key]['method']
        # if(method == 'tdtm'):
        #     ttm = (self.tdtm - 0.5 - dt + 1)/243
        # else:
        #     nextday = CCF.tradingDay[CCF.tradingDay > self.today][dt-1]
        #     ttm = ((
        #         datetime.strptime(self.expiredate, "%Y%m%d")
        #         - datetime.strptime(nextday, "%Y%m%d")
        #     ).days + 0.5)/365
        LGD = LGDratio * (margin)
        portfolioprice = -1 * LGD
        
        # 使用二分查找法找到使得midprice和portfolioprice差值小于1的x值
        lower_bound = 0.1  # 设置下界
        upper_bound = 2  # 设置上界
        
        iteration = 0
        while True:
            iteration += 1
            x = (lower_bound + upper_bound) / 2
            midprice = self.calPortfolioPrice(self.calRELPortInfoInScenrio(dt, x, ivlevel, method))
            
            if abs(midprice - portfolioprice) < 1:
                break
            elif midprice > portfolioprice:
                # 根据组合的delta特性调整搜索范围
                # lotdelta = self.calPortfolioLotDelta(info)
                if self.sell_side == 'p':
                    upper_bound = x
                else:
                    lower_bound = x
            else:
                # 根据组合的delta特性调整搜索范围
                # lotdelta = self.calPortfolioLotDelta(info)
                if self.sell_side == 'p':
                    lower_bound = x
                else:
                    upper_bound = x
                    
            # 如果搜索范围过小，退出循环
            if upper_bound - lower_bound < 0.0001:
                break
        return x * self.und_price
    
    # def backwardScenrioAnalysis(self, info, cost, margin, dt = 1):
    #     ivlevel = [1.05,1.1,1.2,1.3,1.4]
    #     lgdratio = [0.05,0.1,0.2,0.5,0.9]
    #     result = []
    #     for i in ivlevel:
    #         for j in lgdratio:
    #             thisx = self.backwardScenrioAnalysisLevelS(info, cost, margin, i, j, dt)
    #             lgd = j * (margin - cost)
    #             iv = i * info[next(iter(info))]['iv']   
    #             result.append([i, j, thisx, lgd, iv])
    #     result = pd.DataFrame(result, columns=['ivlevel', 'lgdratio', 'und_price', 'lgd', 'iv'])
    #     return result


In [5]:
# date = dt.date.today().strftime("%Y%m%d")
date = '20250515'

accleveldf = pd.read_excel(f"E://ExePort//kurt//{date}//{date}_position.xlsx").iloc[:,1:]

with open(f"E://ExePort//kurt//{date}//{date}_portfoliosFROMEXEPORT.json", 'r') as f:
    portfoliosFROMEXEPORT = json.load(f)
    
# accleveldf['单组开仓价格'] = accleveldf['单组开仓价格']*-1
strategydict = {}
error_lst = []
for stgname in accleveldf['策略名'].unique():
    if(stgname not in portfoliosFROMEXEPORT):
        error_lst.append(stgname)
        raise Exception("策略名无详细信息", stgname)
    portf = portfoliosFROMEXEPORT[stgname]['portfolio']
    try:
        strategydict[stgname] = OptionPortfolio(task_name=stgname, portfolio=portf, today=date)
    except Exception as e:
        print('system error, cannot initialize strategy', stgname, e)
        error_lst.append(stgname)
        continue

ERROR:root:未找到合约 SM507C7300 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: SM507C7300, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 SM507C7300.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=SM507C7300, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3500, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3300, 取close
ERROR:root:初始化期权组合失败: ('Error, portfolio has no basic info', '588000', '20250528')


system error, cannot initialize strategy HX588EO2505:P-800 ('Error, portfolio has no basic info', '588000', '20250528')


ERROR:root:未找到合约 eb2506-P-5900 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: eb2506-P-5900, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 eb2506-P-5900.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=eb2506-P-5900, 取做小tick
ERROR:root:未找到合约 IO2506-P-3100 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: IO2506-P-3100, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 IO2506-P-3100.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=IO2506-P-3100, 取做小tick
ERROR:root:初始化期权组合失败: ('Error, portfolio has no basic info', '588000', '20250625')
ERROR:root:未找到合约 jd2506-C-3700 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: jd2506-C-3700, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 jd2506-C-3700.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=jd2506-C-3700, 取做小tick


system error, cannot initialize strategy HX588EO2506:P-750 ('Error, portfolio has no basic info', '588000', '20250625')


ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3450, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3450, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-P-4900, 取close
ERROR:root:未找到合约 jd2506-C-3700 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: jd2506-C-3700, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 jd2506-C-3700.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=jd2506-C-3700, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-C-7300, 取close
ERROR:root:初始化期权组合失败: ('Error, portfolio has no basic info', '159922', '20250528')


...

system error, cannot initialize strategy SZ500EO2505:PS-1950-1900-1-2 ('Error, portfolio has no basic info', '159922', '20250528')
system error, cannot initialize strategy SZ500EO2505:CS-2550-2600-1-2 ('Error, portfolio has no basic info', '159922', '20250528')


ERROR:root:初始化期权组合失败: ('Error, portfolio has no basic info', '510300', '20250528')
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-C-4450, 取close


system error, cannot initialize strategy 300EO2505:PS-3400-3300-1-3 ('Error, portfolio has no basic info', '510300', '20250528')


ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-C-4300, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-P-3200, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-C-4450, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-C-4350, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-C-4450, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=HO2505-C-3050, 取close
ERROR:root:初始化期权组合失败: ('Error, portfolio has no basic info', '510050', '20250528')


system error, cannot initialize strategy 50EO2505:P-2350 ('Error, portfolio has no basic info', '510050', '20250528')


In [6]:
accleveldf

,账户,策略名,持仓,持仓均价
0,招享1号,sn2506:C-340000,6,-231.666667
1,招享1号,zn2506:P-19400,4,-18.500000
2,招享1号,SM507:C-7300,8,-3.750000
3,招享1号,SA507:CS-1700-1820-1-8,1,-3.000000
4,招享1号,SA507:C-1820,20,-0.500000
...,...,...,...,...
514,星原1号,FG507:C-1420,1,-0.500000
515,星原1号,RM507:P-2000,1,-1.000000
516,星原1号,a2507:P-3400,1,-1.500000
517,星原1号,jd2506:C-3700,1,-1.000000


In [7]:
error_lst

['HX588EO2505:P-800',
 'HX588EO2506:P-750',
 'SZ500EO2505:P-1900',
 'hx588EO2505:PS-900-850-800-1-1-1',
 '588EO2505:P-750',
 'SZ500EO2505:PS-1950-1900-1-2',
 'SZ500EO2505:CS-2550-2600-1-2',
 '300EO2505:PS-3400-3300-1-3',
 '50EO2505:P-2350']

In [8]:
for ind, row in accleveldf.iterrows():
    if(row['策略名'] not in strategydict):
        continue
    strategyobj = strategydict[row['策略名']]
    accleveldf.loc[ind,'当前总占用保证金'] = strategyobj.margin
    accleveldf.loc[ind,'标的合约'] = strategyobj.underlying_instr_id
    accleveldf.loc[ind,'标的品种'] = strategyobj.Undinfo['product']
    accleveldf.loc[ind,'标的板块'] = strategyobj.Undinfo['sector']
    accleveldf.loc[ind,'乘数'] = strategyobj.Undinfo['option_multiple']
    accleveldf.loc[ind,'当前涨跌停板'] = strategyobj.Undinfo['updownlimit']
    accleveldf.loc[ind,'收入到期日'] = strategyobj.expiredate
    if(strategyobj.expiredate <= date):
        continue
    accleveldf.loc[ind,'距离到期交易日'] = strategyobj.tdtm
    accleveldf.loc[ind,'距离到期日历日'] = strategyobj.cdtm
    accleveldf.loc[ind,'标的当日收盘价'] = strategyobj.und_price
    pclose, pask, pbid = strategyobj.calMMPortfolioPrice()
    accleveldf.loc[ind,'单组当日收盘价'] = pclose
    accleveldf.loc[ind,'单组ask'] = pask
    accleveldf.loc[ind,'单组bid'] = pbid
    accleveldf.loc[ind,'单组当前MTM风险比例'] = pclose/strategyobj.margin
    accleveldf.loc[ind,'单组当前对价风险比例'] = pbid/strategyobj.margin
    accleveldf.loc[ind,'当前持仓$损益'] = (pclose-row['持仓均价']*strategyobj.Undinfo['option_multiple']) * row['持仓']
    accleveldf.loc[ind,'当前持仓%损益'] = (pclose-row['持仓均价']*strategyobj.Undinfo['option_multiple']) / strategyobj.margin
    accleveldf.loc[ind,'期初持有到期总$收益'] =row['持仓均价'] * row['持仓'] * strategyobj.Undinfo['option_multiple'] * -1
    accleveldf.loc[ind,'当前持有到期总$收益'] = pclose * row['持仓'] * -1   
    accleveldf.loc[ind,'当前持有到期总%收益'] = pclose / strategyobj.margin * -1
    accleveldf.loc[ind,'当前持有到期总年化%收益'] = pclose /strategyobj.margin * 365 / accleveldf.loc[ind,'距离到期日历日'] * -1
    greeksinfo = strategyobj.calMTMGreeksInfo()
    delta, delta_price, lotdelta = strategyobj.calPortfolioDelta(greeksinfo)
    accleveldf.loc[ind,'单组delta'] = delta
    accleveldf.loc[ind,'单组$delta'] = delta_price
    accleveldf.loc[ind,'单组LotDelta'] = lotdelta
    accleveldf.loc[ind,'总$delta'] = delta_price * row['持仓']
    accleveldf.loc[ind,'总LotDelta'] = lotdelta * row['持仓']
    portf = strategyobj.portfolio
    nsum = sum([abs(i) for i in portf.values()])
    accleveldf.loc[ind,'单组平仓手续费'] = nsum * strategyobj.Undinfo['open_money_by_vol']
    accleveldf.loc[ind,'下一交易日最恶劣情景'] = strategyobj.und_price * (1 + strategyobj.Undinfo['updownlimit']) if strategyobj.sell_side == 'c' else strategyobj.und_price * (1 - strategyobj.Undinfo['updownlimit'])
    #print(accleveldf.loc[ind,'下一交易日最恶劣情景'],row['单组规划保证金'],accleveldf.loc[ind,'标的合约'] )
    accleveldf.loc[ind,'下一交易日最恶劣情景LGD'] = (strategyobj.calPortfolioPrice(strategyobj.calRELPortInfoInScenrio(1, accleveldf.loc[ind,'下一交易日最恶劣情景']/strategyobj.und_price, 1.25, 'tdtm')))/strategyobj.margin
    accleveldf.loc[ind,'VAR'] = accleveldf.loc[ind,'下一交易日最恶劣情景LGD'] * strategyobj.margin * -1 * row['持仓']
    accleveldf.loc[ind,'10LGDwithSameIV'] = strategyobj.backwardScenrioAnalysisLevelS(strategyobj.margin, 0.1)
    accleveldf.loc[ind,'20LGDwithSameIV'] = strategyobj.backwardScenrioAnalysisLevelS(strategyobj.margin, 0.2)
    accleveldf.loc[ind,'50LGDwithSameIV'] = strategyobj.backwardScenrioAnalysisLevelS(strategyobj.margin, 0.5)
    accleveldf.loc[ind,'100LGDwithSameIV'] = strategyobj.backwardScenrioAnalysisLevelS(strategyobj.margin, 1)
    accleveldf.loc[ind,'10LGDwithHigherIV'] = strategyobj.backwardScenrioAnalysisLevelS(strategyobj.margin, 0.1, ivlevel = 1.25)
    accleveldf.loc[ind,'20LGDwithHigherIV'] = strategyobj.backwardScenrioAnalysisLevelS(strategyobj.margin, 0.2, ivlevel = 1.25)
    accleveldf.loc[ind,'50LGDwithHigherIV'] = strategyobj.backwardScenrioAnalysisLevelS(strategyobj.margin, 0.5, ivlevel = 1.25)
    accleveldf.loc[ind,'100LGDwithHigherIV'] = strategyobj.backwardScenrioAnalysisLevelS(strategyobj.margin, 1, ivlevel = 1.25)


e:\SellOption\lib\site-packages\py_vollib\ref_python\black_scholes_merton\__init__.py:87: RuntimeWarning: divide by zero encountered in scalar divide
  return numerator / denominator
e:\SellOption\lib\site-packages\py_vollib\black_scholes_merton\greeks\analytical.py:210: RuntimeWarning: invalid value encountered in scalar divide
  return numerator / denominator
e:\SellOption\lib\site-packages\py_vollib\ref_python\black_scholes_merton\__init__.py:87: RuntimeWarning: divide by zero encountered in scalar divide
  return numerator / denominator
e:\SellOption\lib\site-packages\py_vollib\black_scholes_merton\greeks\analytical.py:210: RuntimeWarning: invalid value encountered in scalar divide
  return numerator / denominator
e:\SellOption\lib\site-packages\py_vollib\ref_python\black_scholes_merton\__init__.py:87: RuntimeWarning: divide by zero encountered in scalar divide
  return numerator / denominator
e:\SellOption\lib\site-packages\py_vollib\black_scholes_merton\greeks\analytical.py:210: 

In [10]:
accleveldf.to_excel(f'E://ExePort//kurt//{date}//{date}_acclevelSummary.xlsx')

In [11]:
accleveldf.dropna(inplace=True)

In [12]:
optionleveldetail = {}
for ind, row in accleveldf.iterrows():
    if(row['策略名'] not in strategydict):
        continue
    strategyobj = strategydict[row['策略名']]
    acct = row['账户']
    if acct not in optionleveldetail:
        optionleveldetail[acct] = {}
    portf = strategyobj.portfolio
    for option_id, shares in portf.items():
        if option_id not in optionleveldetail[acct]:
            optionleveldetail[acct][option_id] = 0
        optionleveldetail[acct][option_id] += shares * row['持仓']
result = []
for acct, holdport in optionleveldetail.items():
    for option_id, shares in holdport.items():
        thisport = {option_id: shares}
        opthis = OptionPortfolio(task_name='temp', portfolio=thisport, today=date)
        future_id = opthis.underlying_instr_id
        prod = opthis.Undinfo['product']
        sec = opthis.Undinfo['sector']
        future_price = opthis.und_price
        future_multiple = opthis.Undinfo['option_multiple']
        expiredate = opthis.expiredate
        tdtm = opthis.tdtm
        cdtdm = opthis.cdtm
        margin = opthis.margin * 1.2
        optype = opthis.sell_side
        multiplier = opthis.Undinfo['option_multiple']
        current_price, ask ,bid = opthis.calMMPortfolioPrice()
        thisinfo = opthis.calMTMGreeksInfo()
        delta, delta_price, lotdelta = opthis.calPortfolioDelta(thisinfo)
        iv = thisinfo[option_id]['iv']
        gamma = thisinfo[option_id]['gamma'] * future_multiple * shares
        theta = thisinfo[option_id]['theta'] * future_multiple * shares
        vega = thisinfo[option_id]['vega'] * future_multiple * shares
        if(shares < 0):
            worstprice = opthis.und_price * (1 + opthis.Undinfo['updownlimit']) if opthis.sell_side == 'c' else opthis.und_price * (1 - opthis.Undinfo['updownlimit'])
            worstlgd = (opthis.calPortfolioPrice(opthis.calRELPortInfoInScenrio(1, worstprice/opthis.und_price, 1.25, 'tdtm')))/margin
            var = worstlgd * margin * -1 
            sameiv10lgd = opthis.backwardScenrioAnalysisLevelS(margin, 0.1)
            sameiv20lgd = opthis.backwardScenrioAnalysisLevelS(margin, 0.2)
            sameiv50lgd = opthis.backwardScenrioAnalysisLevelS(margin, 0.5)
            sameiv100lgd = opthis.backwardScenrioAnalysisLevelS(margin, 1)
            higheriv10lgd = opthis.backwardScenrioAnalysisLevelS(margin, 0.1, ivlevel=1.25)
            higheriv20lgd = opthis.backwardScenrioAnalysisLevelS(margin, 0.2, ivlevel=1.25)
            higheriv50lgd = opthis.backwardScenrioAnalysisLevelS(margin, 0.5, ivlevel=1.25)
            higheriv100lgd = opthis.backwardScenrioAnalysisLevelS(margin, 1, ivlevel=1.25)
        else:
            worstprice = worstlgd = var = sameiv10lgd = sameiv20lgd = sameiv50lgd = sameiv100lgd = higheriv10lgd = higheriv20lgd = higheriv50lgd = higheriv100lgd = np.nan
        
        result.append([acct, option_id, shares, future_id, prod, sec, future_price, expiredate, tdtm, cdtdm, margin, optype, multiplier, current_price, ask, bid, delta, delta_price, lotdelta, iv, gamma, theta, vega, worstprice, worstlgd, var, sameiv10lgd, sameiv20lgd, sameiv50lgd, sameiv100lgd, higheriv10lgd, higheriv20lgd, higheriv50lgd, higheriv100lgd])
result = pd.DataFrame(result, columns=['账户', '期权', '持仓', '标的', '标的品种', '标的板块', '标的价格', '到期日', '到期交易日', '到期日历日', '保证金', '方向', '乘数', '期权组close', '期权组ask', '期权组bid', 'delta', '$delta', 'lotdelta', 'iv', 'gamma', 'theta', 'vega', '最恶劣情景标的收盘价', '最恶劣情景LGDratio', 'VAR', '10LGDwithSameIV', '20LGDwithSameIV', '50LGDwithSameIV', '100LGDwithSameIV', '10LGDwithHigherIV', '20LGDwithHigherIV', '50LGDwithHigherIV', '100LGDwithHigherIV'])


ERROR:root:未找到合约 SM507C7300 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: SM507C7300, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 SM507C7300.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=SM507C7300, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3500, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3300, 取close
ERROR:root:未找到合约 eb2506-P-5900 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: eb2506-P-5900, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 eb2506-P-5900.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=eb2506-P-5900, 取做小tick
ERROR:root:未找到合约 IO2506-P-3100 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: IO2506-P-3100, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 IO2506-P-3100.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=IO2506-P-3100, 取做小tick
ERROR:root:未找到合约 jd2506-C-3700 在日期 20250515 的收盘

In [13]:
result.to_excel(f'E://ExePort//kurt//{date}//{date}_optionLevelSummary.xlsx')

In [15]:
accportlevresult = []

for acc in result["账户"].unique():
    accdf = result[result["账户"] == acc]
    for und in accdf["标的"].unique():
        thisdf = accdf[accdf["标的"] == und]
        prod = thisdf.iloc[0]["标的品种"]
        cdtm = thisdf.iloc[0]["到期日历日"]
        tdtm = thisdf.iloc[0]["到期交易日"]
        undprice = thisdf.iloc[0]["标的价格"]
        marginsum = thisdf["保证金"].sum()
        tottaldelta = thisdf["delta"].sum()
        totaldelta_price = thisdf["$delta"].sum()
        totaldelta_lot = thisdf["lotdelta"].sum()
        totalgamma = thisdf["gamma"].sum()
        totaltheta = thisdf["theta"].sum()
        totalvega = thisdf["vega"].sum()
        maxiv = thisdf["iv"].max()
        expire = thisdf.iloc[0]["到期日"]
        multi = thisdf.iloc[0]["乘数"]
        cdtm = thisdf.iloc[0]["到期日历日"]
        tdtm = thisdf.iloc[0]["到期交易日"]
        sec = thisdf.iloc[0]["标的板块"]
        port = {}
        for ind, row in thisdf.iterrows():
            port[row["期权"]] = row["持仓"]
        try:
            opthis = OptionPortfolio(task_name="temp", portfolio=port, today=date)
        except Exception as e:
            print(f"error in {und} {port} {date}", e)
            continue
        ase = None
        for option_id, detail in opthis.portInfo.items():
            if not prod[0].isalpha():
                if ase is None:
                    ase = (
                        ORME.getPayoffSeries(
                            und,
                            tdtm,
                            undprice,
                            detail["strike_price"],
                            detail["options_type"],
                        )
                        * detail["position"]
                        * multi
                    )
                else:
                    ase = (
                        ase
                        + ORME.getPayoffSeries(
                            und,
                            tdtm,
                            undprice,
                            detail["strike_price"],
                            detail["options_type"],
                        )
                        * detail["position"]
                        * multi
                    )
            else:
                if ase is None:
                    ase = (
                        ORM.getPayoffSeries(
                            prod,
                            tdtm,
                            undprice,
                            detail["strike_price"],
                            detail["options_type"],
                        )
                        * detail["position"]
                        * multi
                    )
                else:
                    ase = (
                        ase
                        + ORM.getPayoffSeries(
                            prod,
                            tdtm,
                            undprice,
                            detail["strike_price"],
                            detail["options_type"],
                        )
                        * detail["position"]
                        * multi
                    )
        var90 = ase.quantile(0.1)
        var95 = ase.quantile(0.05)
        var99 = ase.quantile(0.01)
        var100 = ase.quantile(0)
        var90lgdratio = var90 / marginsum
        var95lgdratio = var95 / marginsum
        var99lgdratio = var99 / marginsum
        var100lgdratio = var100 / marginsum
        accportlevresult.append(
            [
                acc,
                und,
                port,
                sec,
                cdtm,
                tdtm,
                marginsum,
                tottaldelta,
                totaldelta_price,
                totaldelta_lot,
                totalgamma,
                totaltheta,
                totalvega,
                maxiv,
                expire,
                multi,
                var90,
                var95,
                var99,
                var100,
                var90lgdratio,
                var95lgdratio,
                var99lgdratio,
                var100lgdratio,
            ]
        )
accportlevresult = pd.DataFrame(
    accportlevresult,
    columns=[
        "账户",
        "标的",
        "期权组合",
        "标的板块",
        "到期日历日",
        "到期交易日",
        "总保证金",
        "delta",
        "$delta",
        "lotdelta",
        "gamma",
        "theta",
        "vega",
        "maxiv",
        "到期日",
        "乘数",
        "90%var",
        "95%var",
        "99%var",
        "100%var",
        "90%varLGDratio",
        "95%varLGDratio",
        "99%varLGDratio",
        "100%varLGDratio",
    ],
)

ERROR:root:未找到合约 SM507C7300 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: SM507C7300, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 SM507C7300.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=SM507C7300, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3500, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3300, 取close
ERROR:root:未找到合约 jd2506-C-3700 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: jd2506-C-3700, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 jd2506-C-3700.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=jd2506-C-3700, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3450, 取close
ERROR:root:初始化期权组合失败: ('期权组合中存在多个标的和到期日', ['90005442', '90005443', '90005459', '90005460', '90005070', '90005261', '90005260', '90005030', '90004990'])


error in 159915 {'90005442': -59, '90005443': 22, '90005459': 10, '90005460': 10, '90005070': -10, '90005261': -52, '90005260': 26, '90005030': 12, '90004990': -6} 20250515 ('期权组合中存在多个标的和到期日', ['90005442', '90005443', '90005459', '90005460', '90005070', '90005261', '90005260', '90005030', '90004990'])


ERROR:root:未找到合约 eb2506-P-5900 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: eb2506-P-5900, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 eb2506-P-5900.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=eb2506-P-5900, 取做小tick
ERROR:root:未找到合约 IO2506-P-3100 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: IO2506-P-3100, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 IO2506-P-3100.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=IO2506-P-3100, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-P-4900, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-C-7300, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-P-4950, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=HO2505-C-3050, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=HO2505-C-3000, 取close
ERROR:root:未找到合约 b2506-C-4150 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: b2506-C-4150, 日期: 20250515, 错误: <urlopen error ftp error: U

error in 159915 {'90005442': -59, '90005443': 22, '90005459': 10, '90005460': 10, '90005070': -10, '90005261': -52, '90005260': 26, '90005030': 12, '90004990': -6} 20250515 ('期权组合中存在多个标的和到期日', ['90005442', '90005443', '90005459', '90005460', '90005070', '90005261', '90005260', '90005030', '90004990'])


ERROR:root:未找到合约 IO2506-P-3100 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: IO2506-P-3100, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 IO2506-P-3100.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=IO2506-P-3100, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-P-4900, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-C-7300, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-P-4950, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=HO2505-C-3050, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=HO2505-C-3000, 取close
ERROR:root:未找到合约 b2506-C-4150 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: b2506-C-4150, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 b2506-C-4150.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=b2506-C-4150, 取做小tick
ERROR:root:未找到合约 b2506-P-2800 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: b2506-P-2800, 日期: 20250515, 错误: <urlopen error ftp error: URLEr

...

ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-P-4900, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-C-7300, 取close
ERROR:root:未找到合约 b2506-C-4150 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: b2506-C-4150, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 b2506-C-4150.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=b2506-C-4150, 取做小tick
ERROR:root:未找到合约 b2506-P-2800 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: b2506-P-2800, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 b2506-P-2800.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=b2506-P-2800, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-P-3200, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-P-3250, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=i2506-P-600, 取close
ERROR:root:未找到合约 i2506-P-620 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: i2506-P-620, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp

error in 159915 {'90005442': -44, '90005443': 17, '90005459': 8, '90005460': 8, '90005070': -8, '90005261': -40, '90005260': 20, '90005030': 10, '90004990': -5, '90004620': 9, '90004619': 9, '90004617': -9, '90005452': 8, '90004989': -4} 20250515 ('期权组合中存在多个标的和到期日', ['90005442', '90005443', '90005459', '90005460', '90005070', '90005261', '90005260', '90005030', '90004990', '90004620', '90004619', '90004617', '90005452', '90004989'])


ERROR:root:获取期权报价失败 - 合约: IO2506-P-3100, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 IO2506-P-3100.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=IO2506-P-3100, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=HO2505-C-3050, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=HO2505-C-3000, 取close
ERROR:root:未找到合约 b2506-C-4150 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: b2506-C-4150, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 b2506-C-4150.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=b2506-C-4150, 取做小tick
ERROR:root:未找到合约 b2506-P-2800 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: b2506-P-2800, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 b2506-P-2800.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=b2506-P-2800, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-P-3200, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250

error in 159915 {'90005442': -11, '90005443': 3, '90005459': 1, '90005460': 1, '90005070': -1, '90005261': -2, '90005260': 1} 20250515 ('期权组合中存在多个标的和到期日', ['90005442', '90005443', '90005459', '90005460', '90005070', '90005261', '90005260'])


ERROR:root:获取期权报价失败 - 合约: i2506-P-540, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 i2506-P-540.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=i2506-P-540, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-P-3200, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-P-3250, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=MO2505-P-4900, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3500, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=jd2506-C-3300, 取close
ERROR:root:未找到合约 jd2506-C-3700 在日期 20250515 的收盘价
ERROR:root:获取期权报价失败 - 合约: jd2506-C-3700, 日期: 20250515, 错误: <urlopen error ftp error: URLError("ftp error: error_perm('550 jd2506-C-3700.csv: No such file or directory')")>
ERROR:root:无法获取期权数据: 日期=20250515, 合约=jd2506-C-3700, 取做小tick
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-P-3200, 取close
ERROR:root:无法获取期权盘口数据: 日期=20250515, 合约=IO2505-P-3250, 取close


In [16]:
accportlevresult.to_excel(f'E://ExePort//kurt//{date}//{date}_undLevelSummary.xlsx')

In [17]:
def readEXEPORTportfolio():
    sqlcmd = f"""
    SELECT *
    FROM portfolio
    """
    df = pd.read_sql(sql=sqlcmd, con=CCF.EXEPORT_engine)
    result = {}
    for ind, row in df.iterrows():
        name = row['portfolio_name']
        inslst = row['instruments'].split(',')
        poslst = row['trade_ratio'].split(',')
        temp = {}
        for i, j in zip(inslst, poslst):
            temp[i] = int(float(j))*-1
        result[name] = {}
        result[name]['portfolio'] = temp
        result[name]['multiplier'] = row['portfolio_multiple']
        result[name]['recentins'] = sorted(inslst)[0]
    inslst = [i['recentins'] for i in result.values()]
    SQL_cmd = f"""
    SELECT         
        i.instrument_id,
        i.expiredate
    FROM instrument i
    WHERE i.instrument_id IN {tuple(inslst)}
    """
    instrument_info = pd.read_sql(sql=SQL_cmd, con=CCF.market_base)
    instrument_info = instrument_info.set_index('instrument_id')['expiredate'].to_dict()
    for i in result.keys():
        result[i]['expiredate'] = instrument_info[result[i]['recentins']] if result[i]['recentins'] in instrument_info else ''
    return result




In [17]:
stgtreaddict = {}


def readALLSTGTRADE(begin_tag):
    if begin_tag in stgtreaddict:
        return stgtreaddict[begin_tag].copy()
    sqlcmd = f"""
    SELECT operator, traded_time, init_trading_day, fund_name, portfolio_name,stg_tags, direction, volume_traded, traded_price, commission
    FROM stg_trade_raw
    WHERE status = 'DONE' and init_trading_day >= '20250313' and stg_tags like '{begin_tag}%%' 
    """
    df = pd.read_sql(sql=sqlcmd, con=CCF.EXEPORT_engine)
    sqlcmd = f"""
    SELECT operator, traded_time, init_trading_day, fund_name, portfolio_name,stg_tags, direction, volume_traded, traded_price, commission
    FROM stg_trade_raw
    WHERE status = 'DONE' and portfolio_name in {tuple(df['portfolio_name'].unique())}
    """
    df2 = pd.read_sql(sql=sqlcmd, con=CCF.EXEPORT_engine)
    stgtreaddict[begin_tag] = df2
    return df2.copy()

In [18]:
readALLSTGTRADE('kurt')

,operator,traded_time,init_trading_day,fund_name,portfolio_name,stg_tags,direction,volume_traded,traded_price,commission
0,WEN,2025-03-13 11:24:04.592,20250313,招享1号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,1,2.8,0.0
1,WEN,2025-03-13 11:24:04.615,20250313,招享2号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,1,2.8,0.0
2,WEN,2025-03-13 11:24:04.631,20250313,招享3号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,1,2.8,0.0
3,WEN,2025-03-13 11:24:04.653,20250313,招享通胀1号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,3,2.8,0.0
4,WEN,2025-03-13 11:24:04.680,20250313,风险均配1号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,1,2.8,0.0
...,...,...,...,...,...,...,...,...,...,...
2790,WEN,2025-05-19 14:22:23.275,20250519,招享1号,SA507:CS-1580-1820-1-8,kurt_full,1,2,2.5,0.0
2791,WEN,2025-05-19 14:22:23.335,20250519,招享2号,SA507:CS-1580-1820-1-8,kurt_full,1,2,2.5,0.0
2792,WEN,2025-05-19 14:22:23.397,20250519,招享3号,SA507:CS-1580-1820-1-8,kurt_full,1,2,2.5,0.0
2793,WEN,2025-05-19 14:22:23.462,20250519,招享通胀1号,SA507:CS-1580-1820-1-8,kurt_full,1,5,2.5,0.0


In [7]:
def generateDayPosition(date, begin_tag):
    if(date < '20250312'):
        raise Exception('date must be greater than 20250312')
    if(date  == '20250312'):
        temp = pd.DataFrame(columns=['账户', '策略名', '持仓', '持仓均价'])
        path = f"E://ExePort//{begin_tag}//{date}//"
        CCF.mkdir(path)
        temp.to_csv(f"{path}{date}_today_position.csv")
        return temp
    portfoliosFROMEXEPORT = readEXEPORTportfolio()
    allstgtrade = readALLSTGTRADE(begin_tag)
    allstgtrade['trading'] = (allstgtrade['direction'] == 1)*2-1
    allstgtrade['trading'] = allstgtrade['trading']  * allstgtrade['volume_traded'] 
    today_trade = allstgtrade[allstgtrade['init_trading_day'] == date]
    lasttradeday = CCF.tradingDay[CCF.tradingDay < date].max()
    try: 
        lastdaypos = pd.read_csv(f"E://ExePort//{begin_tag}//{lasttradeday}//{lasttradeday}_today_position.csv")
    except:
        lastdaypos = generateDayPosition(lasttradeday, begin_tag)
    lastdaypos['方向'] = '平仓'
    lastdaypos['PNL'] = 0
    todayposdf = pd.concat([lastdaypos, today_trade])
    todayposdf.to_csv(f"E://ExePort//{begin_tag}//{date}//{date}_today_position.csv")
    
    return todayposdf


In [8]:
portfoliosFROMEXEPORT = readEXEPORTportfolio()

def transferToDict(df):
    result = {}
    for ind, row in df.iterrows():
        stgname = row['策略名']
        if(stgname not in result):
            result[stgname] = {}
            result[stgname]['账户'] = row['账户']
            result[stgname]['持仓'] = row['持仓']
            result[stgname]['持仓均价'] = row['持仓均价']
        else:
            prevpos = result[stgname]['持仓']
            thispos = row['持仓']
            result[stgname]['持仓均价'] = (prevpos * result[stgname]['持仓均价'] + thispos * row['持仓均价']) / (prevpos + thispos)
            result[stgname]['持仓'] = prevpos + thispos
    return result

def mergePos(prev_df, today_df, date):
    required_columns = ['账户', '策略名', '持仓', '持仓均价']
    if not all(col in prev_df.columns for col in required_columns):
        raise Exception('prev_df must have columns: 账户, 策略名, 持仓, 持仓均价')
    if not all(col in today_df.columns for col in required_columns):
        raise Exception('today_df must have columns: 账户, 策略名, 持仓, 持仓均价')
    if(len(prev_df['账户'].unique() ) > 1):
        raise Exception('prev_df must have only one account')
    if(len(today_df['账户'].unique()) > 1):
        raise Exception('today_df must have only one account')
    if(len(prev_df) > 0 and len(today_df) > 0):
        if(prev_df['账户'].unique()[0] != today_df['账户'].unique()[0]):
            raise Exception('prev_df and today_df must have the same account')
    prev_dict = transferToDict(prev_df)
    today_trade = transferToDict(today_df)
    today_result = {}
    for stgname in today_trade:
        ### 首先处理trade
        stgdetail = portfoliosFROMEXEPORT[stgname]
        multiplier = stgdetail['multiplier']
        if(stgname not in prev_dict):
            today_result[stgname] = today_trade[stgname].copy()
            today_trade[stgname]['方向'] = '开仓'
            today_trade[stgname]['PNL'] = 0
        else:
            thisstgpos = prev_dict[stgname]['持仓'] + today_trade[stgname]['持仓']
            if (today_trade[stgname]['持仓'] * prev_dict[stgname]['持仓']) > 0:
                # Same direction - update position and average price
                today_result[stgname] = today_trade[stgname].copy()
                today_result[stgname]['持仓'] = thisstgpos
                today_result[stgname]['持仓均价'] = (prev_dict[stgname]['持仓'] * prev_dict[stgname]['持仓均价'] + 
                            today_trade[stgname]['持仓'] * today_trade[stgname]['持仓均价']) /  thisstgpos
                today_trade[stgname]['方向'] = '加仓'
                today_trade[stgname]['PNL'] = 0
            else:
                if(abs(today_trade[stgname]['持仓']) > abs(prev_dict[stgname]['持仓'])):
                    today_result[stgname] = today_trade[stgname].copy()
                    today_result[stgname]['持仓'] = thisstgpos
                    today_result[stgname]['持仓均价'] = today_trade[stgname]['持仓均价']
                    today_trade[stgname]['方向'] = '平昨反开'
                    today_trade[stgname]['PNL'] = (today_trade[stgname]['持仓均价'] - prev_dict[stgname]['持仓均价']) * prev_dict[stgname]['持仓'] * multiplier
                elif(abs(today_trade[stgname]['持仓']) < abs(prev_dict[stgname]['持仓'])):
                    today_result[stgname] = today_trade[stgname].copy()
                    today_result[stgname]['持仓'] = thisstgpos
                    today_result[stgname]['持仓均价'] = prev_dict[stgname]['持仓均价']
                    today_trade[stgname]['方向'] = '部分平仓'
                    today_trade[stgname]['PNL'] = (today_trade[stgname]['持仓均价'] - prev_dict[stgname]['持仓均价']) * today_trade[stgname]['持仓'] * -1 * multiplier
                else:
                    today_trade[stgname]['方向'] = '全平'
                    today_trade[stgname]['PNL'] = (today_trade[stgname]['持仓均价'] - prev_dict[stgname]['持仓均价']) * prev_dict[stgname]['持仓'] * multiplier
    for stgname in prev_dict:
        if stgname not in today_trade:
            stgdetail = portfoliosFROMEXEPORT[stgname]
            multiplier = stgdetail['multiplier']
            expiredate = portfoliosFROMEXEPORT[stgname]['expiredate']
            if(expiredate <= date):
                today_trade[stgname] = prev_dict[stgname].copy()
                today_trade[stgname]['PNL'] = today_trade[stgname]['持仓均价'] * today_trade[stgname]['持仓'] * -1 * multiplier
                today_trade[stgname]['持仓'] = today_trade[stgname]['持仓'] * -1
                today_trade[stgname]['持仓均价'] = 0
                today_trade[stgname]['方向'] = '自然到期'
                continue
            else:
                today_result[stgname] = prev_dict[stgname].copy()
    today_result = pd.DataFrame(today_result).T.reset_index().rename(columns={'index': '策略名'})
    today_trade = pd.DataFrame(today_trade).T.reset_index().rename(columns={'index': '策略名'})
    if(len(today_result) == 0):
        today_result = pd.DataFrame(columns=['账户', '策略名', '持仓', '持仓均价'])
    if(len(today_trade) == 0):
        today_trade = pd.DataFrame(columns=['账户', '策略名', '持仓', '持仓均价','方向','PNL'])
    return today_result, today_trade
    
    
    # prev_df['持仓均价'] = prev_df['持仓均价'].fillna(prev_df['持仓均价'].mean())
    # today_df['持仓均价'] = today_df['持仓均价'].fillna(today_df['持仓均价'].mean())
    # return pd.concat([prev_df, today_df])


In [9]:
# date = '20250514'
portfoliosFROMEXEPORT = readEXEPORTportfolio()
allstgtrade = readALLSTGTRADE('kurt')
allstgtrade = allstgtrade[allstgtrade['portfolio_name'].isin(portfoliosFROMEXEPORT.keys())]
allstgtrade['trading'] = (allstgtrade['direction'] == 1)*2-1
allstgtrade['trading'] = allstgtrade['trading']  * allstgtrade['volume_traded'] 
allstgtrade['traded_price'] = allstgtrade['traded_price'] * -1
# Group by fund_name and sort by traded_time
allstgtrade = allstgtrade.sort_values('traded_time')

# Calculate final position for each fund and portfolio
final_positions = allstgtrade.groupby(['fund_name', 'portfolio_name'])['trading'].sum()

In [27]:
def summaryAllTrade(begin_tag):
    # Get base director
    base_dir = f'E:\\ExePort\\{begin_tag}\\'
    
    # Initialize empty list to store dataframes
    dfs = pd.DataFrame()
    
    # Walk through all subdirectories
    for root, dirs, files in os.walk(base_dir):
        # Find all xlsx files containing 'trade' in filename
        xlsx_files = [f for f in files if f.endswith('.xlsx') and 'trade' in f.lower()]
        
        # Read and append each xlsx file
        for file in xlsx_files:
            file_path = os.path.join(root, file)
            df = pd.read_excel(file_path)
            # Drop first column
            df = df.iloc[:, 1:]
            dfs = pd.concat([dfs, df])
    dfs = dfs.reset_index(drop=True)
    sumdf = dfs.groupby('账户')['PNL'].sum().sort_values(ascending=False).reset_index()
    cm = sns.color_palette("rocket", as_cmap=True)
    a = sumdf.style.background_gradient(cmap=cm, subset="PNL").format(precision=2)
    dfi.export(
        a,
        f"E:\\ExePort\\{begin_tag}\\summary.png",
        fontsize=2,
        dpi=900,
        table_conversion='chrome',
        chrome_path='C:\Program Files\Google\Chrome Dev\Application\chrome.exe',
        max_rows=-1)
    dfs.to_excel(f"E:\\ExePort\\{begin_tag}\\summary.xlsx")
    return dfs


In [16]:
today_trade = allstgtrade[allstgtrade["init_trading_day"] == '20250514']

In [10]:
datelst = CCF.tradingDay[(CCF.tradingDay <= '20250515') & (CCF.tradingDay > '20250312')]
acc_lst = ['招享1号', '招享2号', '招享3号', '招享通胀1号', '风险均配1号', '500指数先锋', '觉醒1号', 'CTA1号', '国债指数先锋', '星原1号']
tempsave = {}
tempdf = allstgtrade[['init_trading_day','fund_name', 'portfolio_name', 'trading', 'traded_price']]
tempdf.columns = ['日期', '账户', '策略名', '持仓', '持仓均价']
for acc in acc_lst:
    tempsave[acc] = {}
    for i in range(len(datelst)):
        today = datelst[i]
        acctodaytrade = tempdf[(tempdf['账户'] == acc) & (tempdf['日期'] == today)]
        if(len(acctodaytrade) == 0):
            acctodaytrade = pd.DataFrame(columns=['账户', '策略名', '持仓', '持仓均价'])
        if(i == 0):
            tempsave[acc][today] = mergePos(pd.DataFrame(columns=['账户', '策略名', '持仓', '持仓均价']), acctodaytrade, today)
        else:
            lastday = datelst[i-1]
            lastdaypos = tempsave[acc][lastday][0]
            tempsave[acc][today] = mergePos(lastdaypos, acctodaytrade, today)



In [12]:
tempsave['招享1号']['20250515'][1]

,账户,策略名,持仓,持仓均价,方向,PNL


In [44]:
posalldf = pd.DataFrame()
tradealldf = pd.DataFrame()
for acc in acc_lst:
    for today in datelst:
        postemp = tempsave[acc][today][0]
        tradetemp = tempsave[acc][today][1]
        postemp['日期'] = today
        tradetemp['日期'] = today
        posalldf = pd.concat([posalldf, postemp])
        tradealldf = pd.concat([tradealldf, tradetemp])
tradealldf.rename(columns={'持仓均价': '交易价格', '持仓': '交易手数'}, inplace=True)
posalldf.to_excel('./temp/posalldf.xlsx')
tradealldf.to_excel('./temp/tradealldf.xlsx')



In [59]:
temp = pd.read_excel(f'E:\\ExePort\\kurt\\20250421\\20250421_trade.xlsx').iloc[:, 1:]

temp = temp.groupby(['策略名', '账户']).agg({
    '交易手数': 'sum',
    '交易价格': 'mean',
    'PNL': 'sum'
})
temp


交易手数       交易价格  PNL
策略名                        账户                           
915EO2506:PS-1850-1750-1-2 500指数先锋    -2   0.003000    0
                           招享1号       -4   0.003325    0
                           招享2号       -4   0.003325    0
                           招享3号       -4   0.003300    0
                           觉醒1号       -3   0.003133    0
...                                  ...        ...  ...
sn2505:P-210000            招享2号        4 -15.000000    0
                           招享3号        4 -15.000000    0
                           招享通胀1号     14 -14.857143    0
                           觉醒1号        4 -15.000000    0
                           风险均配1号      4 -15.000000    0

[72 rows x 3 columns]

In [53]:
temp['策略名'].unique()


array(['RM507:P-2000', 'b2506:P-2800', 'sn2505:C-340000',
       'IO2505:PS-3250-3200-1-2', '915EO2506:PS-1850-1750-1-2',
       'au2505:P-600', 'au2505:C-920', 'jd2506:C-3700', 'sn2505:P-210000',
       'SH506:C-3600', 'MO2506:PS-5500-5200-1-2', 'si2506:C-12000'],
      dtype=object)

In [43]:
aa = tempsave['招享1号']['20250512'][1]
aa

,策略名,账户,持仓,持仓均价,方向,PNL
0,jd2506:CS-3300-3500-1-5,招享1号,2,-1.5,开仓,0
1,ni2506:CS-1460-1540-1-3,招享1号,2,-94.0,开仓,0
2,915EO2505:PS-1650-1600-1-7,招享1号,3,-0.0028,开仓,0


In [21]:
for ind, row in aa.iterrows():
    stgname = row['策略名']
    if(row['持仓']  != final_positions['招享1号'][stgname]):
        print(stgname)


In [30]:
for key in portfoliosFROMEXEPORT:
    if key[:5] == 'FG506':
        print(key)


FG506:C-1700
FG506:C-1580
FG506:C-1540
FG506:C-1520
FG506:C-1620
FG506:C-1680
FG506:C-1400
FG506:CS-1400-1680-1-9
FG506:C-1300


In [31]:
portfoliosFROMEXEPORT['FG506:C-1700']


{'portfolio': {'FG506C1700': -1},
 'multiplier': 20,
 'recentins': 'FG506C1700',
 'expiredate': '20250513'}

In [15]:
final_positions['招享1号']['']

np.int64(0)

In [16]:
allstgtrade[(allstgtrade['portfolio_name'] == 'FG509:PS-1080-1020-1000-1-1-1') & (allstgtrade['fund_name'] == '招享1号')]

,operator,traded_time,init_trading_day,fund_name,portfolio_name,stg_tags,direction,volume_traded,traded_price,commission,trading
1689,WEN,2025-04-29 14:46:24.379,20250429,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_full,0,1,-1.0,0.0,-1
1697,WEN,2025-04-29 14:46:26.037,20250429,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_full,0,1,-1.0,0.0,-1
1704,WEN,2025-04-29 14:46:27.962,20250429,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_full,0,1,-1.0,0.0,-1
1712,WEN,2025-04-29 14:46:32.152,20250429,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_full,0,1,-1.0,0.0,-1
1720,WEN,2025-04-29 14:47:07.183,20250429,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_full,0,1,-1.0,0.0,-1
2335,WEN,2025-05-09 13:42:37.986,20250509,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_close,1,1,-3.5,0.0,1
2343,WEN,2025-05-09 13:42:39.840,20250509,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_close,1,1,-3.5,0.0,1
2350,WEN,2025-05-09 13:42:40.863,20250509,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_close,1,1,-3.5,0.0,1
2358,WEN,2025-05-09 13:42:42.860,20250509,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_close,1,1,-3.5,0.0,1
2366,WEN,2025-05-09 13:42:45.082,20250509,招享1号,FG509:PS-1080-1020-1000-1-1-1,kurt_close,1,1,-3.5,0.0,1


In [23]:
testdf = allstgtrade[(allstgtrade['portfolio_name'] == 'FG509:PS-1080-1020-1000-1-1-1') & (allstgtrade['fund_name'] == '招享1号')][['fund_name','init_trading_day','portfolio_name','traded_price','volume_traded']]
testdf.columns = ['账户', '日期', '策略名', '持仓均价', '持仓']
def transferToDict(df):
    result = {}
    for ind, row in df.iterrows():
        stgname = row['策略名']
        if(stgname not in result):
            result[stgname] = {}
            result[stgname]['账户'] = row['账户']
            result[stgname]['持仓'] = row['持仓']
            result[stgname]['持仓均价'] = row['持仓均价']
        else:
            prevpos = result[stgname]['持仓']
            thispos = row['持仓']
            result[stgname]['持仓均价'] = (prevpos * result[stgname]['持仓均价'] + thispos * row['持仓均价']) / (prevpos + thispos)
            result[stgname]['持仓'] = prevpos + thispos
    return result

transferToDict(testdf)


{'FG509:PS-1080-1020-1000-1-1-1': {'账户': '招享1号', '持仓': 10, '持仓均价': -2.25}}

In [24]:
testdf

,账户,日期,策略名,持仓均价,持仓
1689,招享1号,20250429,FG509:PS-1080-1020-1000-1-1-1,-1.0,1
1697,招享1号,20250429,FG509:PS-1080-1020-1000-1-1-1,-1.0,1
1704,招享1号,20250429,FG509:PS-1080-1020-1000-1-1-1,-1.0,1
1712,招享1号,20250429,FG509:PS-1080-1020-1000-1-1-1,-1.0,1
1720,招享1号,20250429,FG509:PS-1080-1020-1000-1-1-1,-1.0,1
2335,招享1号,20250509,FG509:PS-1080-1020-1000-1-1-1,-3.5,1
2343,招享1号,20250509,FG509:PS-1080-1020-1000-1-1-1,-3.5,1
2350,招享1号,20250509,FG509:PS-1080-1020-1000-1-1-1,-3.5,1
2358,招享1号,20250509,FG509:PS-1080-1020-1000-1-1-1,-3.5,1
2366,招享1号,20250509,FG509:PS-1080-1020-1000-1-1-1,-3.5,1


In [92]:
final_positions['招享1号']['sn2506:C-340000']


np.int64(6)

In [13]:
date = '20250512'
portfoliosFROMEXEPORT = readEXEPORTportfolio()
allstgtrade = readALLSTGTRADE('kurt')
allstgtrade['trading'] = (allstgtrade['direction'] == 1)*2-1
allstgtrade['trading'] = allstgtrade['trading']  * allstgtrade['volume_traded'] 
# Group by fund_name and sort by traded_time
allstgtrade = allstgtrade.sort_values('traded_time')

# Calculate final position for each fund and portfolio
final_positions = allstgtrade.groupby(['fund_name', 'portfolio_name'])['trading'].sum()

In [10]:
portfoliosFROMEXEPORT

{'IH:05-09': {'portfolio': {'IH2505': -1, 'IH2509': 1},
  'multiplier': 300,
  'recentins': 'IH2505',
  'expiredate': '20250516'},
 'IH:05-12': {'portfolio': {'IH2505': -1, 'IH2512': 1},
  'multiplier': 300,
  'recentins': 'IH2505',
  'expiredate': '20250516'},
 'IH:06-12': {'portfolio': {'IH2506': -1, 'IH2512': 1},
  'multiplier': 300,
  'recentins': 'IH2506',
  'expiredate': '20250620'},
 'IH:09-12': {'portfolio': {'IH2509': -1, 'IH2512': 1},
  'multiplier': 300,
  'recentins': 'IH2509',
  'expiredate': '20250919'},
 'IH:05-06': {'portfolio': {'IH2505': -1, 'IH2506': 1},
  'multiplier': 300,
  'recentins': 'IH2505',
  'expiredate': '20250516'},
 'IH:06-09': {'portfolio': {'IH2506': -1, 'IH2509': 1},
  'multiplier': 300,
  'recentins': 'IH2506',
  'expiredate': '20250620'},
 'IC:05-06-09': {'portfolio': {'IC:05-06': -1, 'IC:06-09': 1},
  'multiplier': 200,
  'recentins': 'IC:05-06',
  'expiredate': ''},
 'IF:05-09-12': {'portfolio': {'IF:05-09': -1, 'IF:09-12': 1},
  'multiplier': 300

In [9]:
allstgtrade

,operator,traded_time,init_trading_day,fund_name,portfolio_name,stg_tags,direction,volume_traded,traded_price,commission,trading
0,WEN,2025-03-13 11:24:04.592,20250313,招享1号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,1,2.80,0.000000,1
1,WEN,2025-03-13 11:24:04.615,20250313,招享2号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,1,2.80,0.000000,1
2,WEN,2025-03-13 11:24:04.631,20250313,招享3号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,1,2.80,0.000000,1
3,WEN,2025-03-13 11:24:04.653,20250313,招享通胀1号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,3,2.80,0.000000,3
4,WEN,2025-03-13 11:24:04.680,20250313,风险均配1号,MO2506:P-3900,"Open,FH,SV,SINGLE",1,1,2.80,0.000000,1
...,...,...,...,...,...,...,...,...,...,...,...
2583,WEN,2025-05-14 14:58:42.443,20250514,招享1号,FG507:C-1400,kurt_income,1,4,0.50,0.000000,4
2584,WEN,2025-05-14 14:58:42.482,20250514,招享通胀1号,FG507:C-1400,kurt_income,1,16,0.50,0.000000,16
2585,WEN,2025-05-14 14:58:42.521,20250514,招享2号,FG507:C-1400,kurt_income,1,4,0.50,0.000000,4
2586,WEN,2025-05-14 14:58:42.554,20250514,招享3号,FG507:C-1400,kurt_income,1,4,0.50,0.000000,4


In [6]:
final_positions[('国债指数先锋', 'FG506:C-1700')]

np.int64(8)

In [16]:
type(final_positions)

pandas.core.series.Series

In [33]:
portfoliosFROMEXEPORT['300EO2504:P-3300']

KeyError: '300EO2504:P-3300'

In [6]:
today_position = []
for i in final_positions.index:
    acc = i[0]
    stgname = i[1]
    if(stgname not in portfoliosFROMEXEPORT):
        logging.error(f'{stgname} not in portfoliosFROMEXEPORT')
        continue
    if(portfoliosFROMEXEPORT[stgname]['expiredate'] <= date):
        continue
    if(final_positions[i] == 0):
        continue
    if(acc not in today_position):
        today_position.append([acc, stgname, int(final_positions[i])])


ERROR:root:300EO2504:P-3300 not in portfoliosFROMEXEPORT
ERROR:root:300EO:2504-P-3300 not in portfoliosFROMEXEPORT
ERROR:root:300EO2504:P-3300 not in portfoliosFROMEXEPORT
ERROR:root:300EO:2504-P-3300 not in portfoliosFROMEXEPORT
ERROR:root:300EO2504:P-3300 not in portfoliosFROMEXEPORT
ERROR:root:300EO:2504-P-3300 not in portfoliosFROMEXEPORT
ERROR:root:MO2504:P-5000 not in portfoliosFROMEXEPORT
ERROR:root:300EO2504:P-3300 not in portfoliosFROMEXEPORT
ERROR:root:300EO:2504-P-3300 not in portfoliosFROMEXEPORT
ERROR:root:MO2504:P-5000 not in portfoliosFROMEXEPORT
ERROR:root:300EO2504:P-3300 not in portfoliosFROMEXEPORT
ERROR:root:300EO:2504-P-3300 not in portfoliosFROMEXEPORT
ERROR:root:MO2504:P-5000 not in portfoliosFROMEXEPORT
ERROR:root:MO2504:P-5000 not in portfoliosFROMEXEPORT
ERROR:root:300EO2504:P-3300 not in portfoliosFROMEXEPORT
ERROR:root:300EO:2504-P-3300 not in portfoliosFROMEXEPORT
ERROR:root:MO2504:P-5000 not in portfoliosFROMEXEPORT
ERROR:root:MO2504:P-5000 not in portfoli

In [7]:
pd.DataFrame(today_position, columns=['账户', '策略名', '持仓']).to_excel('./temp/today_position.xlsx')

In [7]:
a['zn2505:P-19600']

{'zn2505P19600': -1}

In [9]:
try:
    OptionPortfolio('temp', a['zn2505:P-19600'], '20250512')
except Exception as e:
    if str(e).startswith("('Error, portfolio has no basic info'"):
        print("Portfolio basic info not found - skipping this portfolio")
    else:
        raise

ERROR:root:初始化期权组合失败: ('Error, portfolio has no basic info', 'zn2505', '20250424')


Portfolio basic info not found - skipping this portfolio


In [18]:
begin_tag = 'kurt'
date = '20250519'

In [20]:
port = readEXEPORTportfolio()
df = pd.read_excel('E://ExePort\kurt//20250519//20250519_trade.xlsx').iloc[:,1:]

In [22]:
port

{'IH:07-09': {'portfolio': {'IH2507': -1, 'IH2509': 1},
  'multiplier': 300,
  'recentins': 'IH2507',
  'expiredate': '20250718'},
 'IH:07-12': {'portfolio': {'IH2507': -1, 'IH2512': 1},
  'multiplier': 300,
  'recentins': 'IH2507',
  'expiredate': '20250718'},
 'IH:06-12': {'portfolio': {'IH2506': -1, 'IH2512': 1},
  'multiplier': 300,
  'recentins': 'IH2506',
  'expiredate': '20250620'},
 'IH:09-12': {'portfolio': {'IH2509': -1, 'IH2512': 1},
  'multiplier': 300,
  'recentins': 'IH2509',
  'expiredate': '20250919'},
 'IH:06-07': {'portfolio': {'IH2506': -1, 'IH2507': 1},
  'multiplier': 300,
  'recentins': 'IH2506',
  'expiredate': '20250620'},
 'IH:06-09': {'portfolio': {'IH2506': -1, 'IH2509': 1},
  'multiplier': 300,
  'recentins': 'IH2506',
  'expiredate': '20250620'},
 'IC:06-07-09': {'portfolio': {'IC:06-07': -1, 'IC:07-09': 1},
  'multiplier': 200,
  'recentins': 'IC:06-07',
  'expiredate': ''},
 'IF:07-09-12': {'portfolio': {'IF:07-09': -1, 'IF:09-12': 1},
  'multiplier': 300

In [24]:
df

,账户,策略名,方向,交易手数,交易价格,PNL,交易额
0,招享1号,ao2506:CS-3400-3500-1-2,开仓,1,-5.0,0,-100.0
1,招享1号,i2507:P-600,开仓,1,-0.3,0,-30.0
2,招享1号,SA507:C-1820,加仓,31,-0.5,0,-310.0
3,招享1号,SA507:C-1620,加仓,-4,-1.0,0,80.0
4,招享1号,SH507:CS-3000-3080-1-2,开仓,1,-2.5,0,-75.0
...,...,...,...,...,...,...,...
64,觉醒1号,sn2506:C-340000,加仓,2,-26.0,0,-52.0
65,觉醒1号,sn2506:P-210000,加仓,2,-28.0,0,-56.0
66,觉醒1号,sc2507:CS-550-570-1-3,加仓,1,-0.5,0,-500.0
67,星原1号,cs2507:P-2300,开仓,2,-0.5,0,-10.0


In [23]:
df['交易额'] = df['交易价格'] * df['交易手数'] * df['策略名'].apply(lambda x: port[x]['multiplier'])

In [39]:
pivot_table = pd.pivot_table(
    df,
    values=['交易手数', '交易价格', '交易额'],
    index=['账户', '方向', '策略名'],
    aggfunc={
        '交易手数': 'sum',
        '交易价格': 'mean',
        '交易额': 'sum'
    }
)
sumtradev = str(int(df['交易额'].sum()))
sumtradev = f'账户交易额({sumtradev})'
# Add column sum for 交易额 by 账户, but only show in first row of each 账户
pivot_table[sumtradev] = pivot_table.groupby('账户')['交易额'].transform('sum')
pivot_table[sumtradev] = pivot_table[sumtradev].mask(pivot_table.groupby('账户').cumcount() > 0)
pivot_table

交易价格  交易手数    交易额  账户交易额(-10210)
账户      方向 策略名                                                       
500指数先锋 全平 au2506:P-600             -0.06    -1   60.0           60.0
招享1号    加仓 SA507:C-1620             -1.00    -4   80.0        -1333.0
           SA507:C-1820             -0.50    31 -310.0            NaN
           sc2507:CS-550-570-1-3    -0.50     1 -500.0            NaN
           sn2506:C-340000         -26.00     2  -52.0            NaN
...                                   ...   ...    ...            ...
风险均配1号  开仓 SA507:CS-1640-1820-1-6   -2.00     3 -120.0            NaN
           SH507:CS-3000-3080-1-2   -2.50     1  -75.0            NaN
           ao2506:CS-3400-3500-1-2  -5.00     1 -100.0            NaN
           cs2507:P-2300            -0.50     5  -25.0            NaN
           i2507:P-600              -0.30     1  -30.0            NaN

[69 rows x 4 columns]

In [45]:
temp = pd.read_excel("E://ExePort//kurt//20250515//20250515_acclevelSummary.xlsx").iloc[:,1:]
with pd.ExcelWriter("E://ExePort//1.xlsx", engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    temp.to_excel(writer, sheet_name='rawdata')

In [16]:
pivot_table = pd.pivot_table(
    df,
    values=['交易手数', '交易价格', 'PNL'],
    index=['方向', '账户', '策略名'],
    aggfunc={
        '交易手数': 'sum',
        '交易价格': 'mean',
        'PNL': 'sum'
    }
).sort_values(('PNL'), ascending=False)
path = f"E://ExePort//{begin_tag}//{date}//picfile//"
CCF.mkdir(path)
dfi.export(
    pivot_table,
    f"{path}todaytradf.png",
    fontsize=2,
    dpi=900,
    table_conversion='chrome',
    chrome_path='C:\Program Files\Google\Chrome Dev\Application\chrome.exe',
    max_rows=-1)

E://ExePort//kurt//20250519//picfile//  目录已存在
